In [ ]:
%%bash
pip install numpy scipy matplotlib pandas torch scikit-learn --quiet

In [ ]:
import os
import glob
import struct
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import resample as scipy_resample
from sklearn.metrics import roc_auc_score, f1_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
print('Imports OK')

In [ ]:
DATA_DIR    = r'C:/Users/quack/Documents/Projects/Verus/Data/Stephen Terracon Cornbread/Data'
WORKING_DIR = r'C:/Users/quack/Documents/Projects/Verus/verus/server/models'
TARGET_SAMPLES = 512
BATCH_SIZE  = 512
EPOCHS      = 100
PATIENCE    = 15
LR          = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
def preprocess_trace(raw_trace, target_samples=512):
    '''
    Standardize any GPR trace to 512 samples, normalized.
    Format-agnostic: works for GSSI, Proceq, any system.
    Returns float32 array shape (512,)
    '''
    if len(raw_trace) != target_samples:
        raw_trace = scipy_resample(raw_trace, target_samples)
    raw_trace = raw_trace - raw_trace.mean()
    max_abs = np.abs(raw_trace).max()
    if max_abs > 0:
        raw_trace = raw_trace / max_abs
    return raw_trace.astype(np.float32)


def preprocess_batch(raw_traces, target_samples=512):
    '''Vectorized preprocessing: (N, L) -> (N, target_samples) float32.'''
    if raw_traces.shape[1] != target_samples:
        raw_traces = scipy_resample(raw_traces, target_samples, axis=1)
    raw_traces = raw_traces - raw_traces.mean(axis=1, keepdims=True)
    norms = np.abs(raw_traces).max(axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return (raw_traces / norms).astype(np.float32)


def get_corrosion_label(cscan_amp_value, p25, p75):
    '''
    Binary corrosion label from CScan amplitude.
    Low amplitude (below 25th percentile) = high risk = label 1
    High amplitude (above 75th percentile) = low risk = label 0
    Middle 50% excluded from training (ambiguous zone) = label -1

    Note: CScan pseudo-labels capture relative amplitude patterns,
    not absolute corrosion state. Targets (AUC>0.70, Sens>60%) are
    intentionally modest because of this label noise.
    '''
    if cscan_amp_value < p25:
        return 1   # high risk
    elif cscan_amp_value > p75:
        return 0   # low risk
    else:
        return -1  # exclude (ambiguous)

In [ ]:
# Proceq RIS .scan binary format constants
_MAGIC        = b'VH01SW'
_D_START      = 0x027C
_D_SIZE       = 0x040C   # 1036 bytes per D-block
_D_HEADER     = 16
_D_SAMPLES    = (_D_SIZE - _D_HEADER) // 2  # 510
_D_REF_BLOCKS = 16
_D_MARKER     = b'D' + bytes(1)             # b'D\x00'

# CScan header: 4*uint32 (version, n_cols, n_cross, n_depth)
#              + 4*float32 (dx_m, depth_range, dy_m, reserved)
_CSCAN_HEADER_BYTES = 32


def read_proceq_traces(scan_path):
    '''Read odd-numbered Proceq RIS PRC .scan file. Returns (traces (N,510), n_traces).'''
    with open(scan_path, 'rb') as f:
        raw = f.read()
    if raw[:6] != _MAGIC:
        return None, 0

    n_total = 0
    pos = _D_START
    while pos + 2 <= len(raw) and raw[pos:pos+2] == _D_MARKER:
        n_total += 1
        pos += _D_SIZE

    if n_total == 0:
        return None, 0
    n_data = n_total - _D_REF_BLOCKS
    if n_data <= 0:
        return None, 0

    data_start = _D_START + _D_REF_BLOCKS * _D_SIZE
    traces = np.zeros((n_data, _D_SAMPLES), dtype=np.float32)
    with open(scan_path, 'rb') as f:
        f.seek(data_start)
        for i in range(n_data):
            block = f.read(_D_SIZE)
            if len(block) < _D_SIZE:
                traces = traces[:i]
                break
            s = np.frombuffer(block[_D_HEADER:], dtype='<i2').astype(np.float32)
            s -= s.mean()
            traces[i] = s

    norms = np.abs(traces).max(axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    traces /= norms
    return traces, len(traces)


def read_cscan_amplitude(cscan_path):
    '''
    Read Proceq .CScan file, return mean peak amplitude across cross-track channels.
    Returns along-track amplitude array shape (n_cols,), int16 scale.
    CScan data is int16 (signed); abs gives 0-32767 range.
    '''
    with open(cscan_path, 'rb') as f:
        raw = f.read()
    _, n_cols, n_cross, n_depth = struct.unpack_from('<4I', raw, 0)
    expected = n_cols * n_cross * n_depth * 2  # int16 = 2 bytes
    data = np.frombuffer(raw[_CSCAN_HEADER_BYTES:_CSCAN_HEADER_BYTES + expected],
                         dtype=np.int16)
    data = data.reshape(n_depth, n_cross, n_cols).astype(np.float32)
    peak_amp  = np.abs(data).max(axis=0)         # (n_cross, n_cols) â€” max across depth
    along_amp = peak_amp.mean(axis=0)            # (n_cols,) â€” mean across cross-track
    return along_amp, n_cols

In [ ]:
def load_corrosion_dataset():
    scan_files   = sorted(glob.glob(os.path.join(DATA_DIR, 'PRC_*.scan')))
    odd_scans    = [
        f for f in scan_files
        if int(os.path.basename(f).replace('PRC_', '').replace('.scan', '')) % 2 == 1
    ]
    # CScans may be in subdirectories (swath_*/CScan_01.CScan)
    cscan_files  = sorted(glob.glob(os.path.join(DATA_DIR, '**', 'CScan_01.CScan'), recursive=True))
    if not cscan_files:
        cscan_files = sorted(glob.glob(os.path.join(DATA_DIR, 'CScan_01.CScan')))
    n_swaths = min(len(odd_scans) // 4, len(cscan_files))
    print(f'Odd scans: {len(odd_scans)}, CScan files: {len(cscan_files)}, swaths: {n_swaths}')

    # First pass: load all CScan amplitudes for global percentile computation
    print('Loading CScan files for global normalization...')
    cscan_amps = []
    for cscan_path in cscan_files[:n_swaths]:
        amp, n_cols = read_cscan_amplitude(cscan_path)
        cscan_amps.append(amp)
        print(f'  {os.path.basename(os.path.dirname(cscan_path)):20s}  '
              f'n_cols={n_cols}  amp range [{amp.min():.0f}, {amp.max():.0f}]')

    all_amp_flat = np.concatenate(cscan_amps)
    p25 = float(np.percentile(all_amp_flat, 25))
    p75 = float(np.percentile(all_amp_flat, 75))
    print(f'\nGlobal amplitude p25={p25:.1f}, p75={p75:.1f}')

    # Second pass: build trace dataset with pseudo-labels
    all_traces, all_labels, all_swath_ids = [], [], []
    n_excluded = 0
    class_counts = {0: 0, 1: 0}

    for sw_idx in range(n_swaths):
        swath_scans = odd_scans[sw_idx * 4 : sw_idx * 4 + 4]
        amp_along   = cscan_amps[sw_idx]   # (n_cols,)

        for scan_path in swath_scans:
            raw_traces, n_data = read_proceq_traces(scan_path)
            if raw_traces is None or n_data < 10:
                continue
            N = len(raw_traces)

            # Interpolate CScan amplitude from n_cols -> N traces
            x_src        = np.linspace(0, N - 1, len(amp_along))
            amp_per_trace = np.interp(np.arange(N), x_src, amp_along)

            # Assign pseudo-labels: 0=healthy, 1=at-risk, -1=exclude
            labels = np.where(
                amp_per_trace < p25, 1,
                np.where(amp_per_trace > p75, 0, -1)
            ).astype(np.int32)

            valid_mask  = labels >= 0
            n_excluded += int((~valid_mask).sum())
            if valid_mask.sum() == 0:
                continue

            processed = preprocess_batch(raw_traces[valid_mask], TARGET_SAMPLES)
            valid_labels = labels[valid_mask].astype(np.float32)
            all_traces.append(processed)
            all_labels.append(valid_labels)
            all_swath_ids.extend([sw_idx] * valid_mask.sum())
            for lbl in [0, 1]:
                class_counts[lbl] += int((valid_labels == lbl).sum())

    print(f'\nExcluded (ambiguous): {n_excluded:,}')
    print(f'Class distribution after exclusion:')
    total_valid = sum(class_counts.values())
    for lbl, name in [(0, 'healthy (0)'), (1, 'at-risk (1)')]:
        pct = 100 * class_counts[lbl] / max(total_valid, 1)
        print(f'  {name}: {class_counts[lbl]:,}  ({pct:.1f}%)')

    return (all_traces, all_labels, np.array(all_swath_ids),
            n_swaths, p25, p75, class_counts)


(swath_traces, swath_labels, swath_ids,
 n_swaths, p25, p75, class_counts) = load_corrosion_dataset()

all_traces = np.concatenate(swath_traces)
all_labels = np.concatenate(swath_labels)

# Split by swath: 0-10 train, 11-13 val
train_mask = swath_ids < 11
val_mask   = swath_ids >= 11

X_train, y_train = all_traces[train_mask], all_labels[train_mask]
X_val,   y_val   = all_traces[val_mask],   all_labels[val_mask]

print(f'\nn_train_traces: {len(X_train):,}')
print(f'n_val_traces:   {len(X_val):,}')

In [ ]:
class GPRCorrosionDataset(Dataset):
    def __init__(self, traces, labels, augment=False):
        self.traces  = torch.from_numpy(traces).unsqueeze(1)  # (N, 1, 512)
        self.labels  = torch.from_numpy(labels)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = self.traces[idx].clone()
        y = self.labels[idx]
        if self.augment:
            if torch.rand(1) < 0.5:
                x = x + torch.randn_like(x) * 0.01
            if torch.rand(1) < 0.5:
                x = x * (0.9 + torch.rand(1) * 0.2)
            if torch.rand(1) < 0.5:
                shift = torch.randint(-10, 11, (1,)).item()
                x = torch.roll(x, shift, dims=-1)
                if shift > 0:
                    x[..., :shift] = 0.0
                elif shift < 0:
                    x[..., shift:] = 0.0
        return x, y


train_ds = GPRCorrosionDataset(X_train, y_train, augment=True)
val_ds   = GPRCorrosionDataset(X_val,   y_val,   augment=False)

# WeightedRandomSampler for balanced batches
n0 = int((y_train == 0).sum())
n1 = int((y_train == 1).sum())
w0 = 1.0 / max(n0, 1)
w1 = 1.0 / max(n1, 1)
sample_weights = np.where(y_train == 1, w1, w0)
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_ds),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,     num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')
print(f'Class weights  0 (healthy): {w0:.6f}  1 (at-risk): {w1:.6f}')

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score = nn.Linear(channels, 1)

    def forward(self, x):
        w = torch.softmax(self.score(x.transpose(1, 2)), dim=1)
        return (x.transpose(1, 2) * w).sum(dim=1)


class CorrosionCNN(nn.Module):
    '''
    Corrosion risk binary classifier.
    Input:  (batch, 1, 512) normalized trace
    Output: (batch,) logit â€” apply sigmoid for probability.
    NO sigmoid in forward â€” use BCEWithLogitsLoss during training.
    '''
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,   32,  7, padding=3), nn.ReLU(),
            nn.MaxPool1d(2),                               # 512->256
            nn.Conv1d(32,  64,  5, padding=2), nn.ReLU(),
            nn.MaxPool1d(2),                               # 256->128
            nn.Conv1d(64,  128, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),                               # 128->64
            nn.Conv1d(128, 128, 3, padding=1), nn.ReLU(),
            nn.MaxPool1d(2),                               # 64->32
        )
        self.attn = TemporalAttention(128)
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            # No sigmoid here â€” BCEWithLogitsLoss is numerically stable
        )

    def forward(self, x):
        return self.head(self.attn(self.conv(x))).squeeze(-1)


model = CorrosionCNN().to(DEVICE)
print(f'CorrosionCNN parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

best_auc       = 0.0
patience_count = 0
history        = []

print('  Ep   TR_loss  Val_loss   Val_AUC   Val_F1    Sens%    FPR%          LR')
print('-' * 79)

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * len(yb)
    tr_loss /= len(train_ds)

    model.eval()
    val_loss, logits_v, tgts_v = 0.0, [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logit  = model(xb)
            val_loss += criterion(logit, yb).item() * len(yb)
            logits_v.append(logit.cpu().numpy())
            tgts_v.append(yb.cpu().numpy())
    val_loss /= len(val_ds)
    logits_v = np.concatenate(logits_v)
    tgts_v   = np.concatenate(tgts_v)
    probs_v  = 1.0 / (1.0 + np.exp(-logits_v))  # sigmoid
    preds_v  = (probs_v >= 0.5).astype(int)

    auc = roc_auc_score(tgts_v, probs_v)
    f1  = f1_score(tgts_v, preds_v, zero_division=0)

    # Sensitivity = TP / (TP + FN), FPR = FP / (FP + TN)
    tp = float(((preds_v == 1) & (tgts_v == 1)).sum())
    fn = float(((preds_v == 0) & (tgts_v == 1)).sum())
    fp = float(((preds_v == 1) & (tgts_v == 0)).sum())
    tn = float(((preds_v == 0) & (tgts_v == 0)).sum())
    sens = 100.0 * tp / max(tp + fn, 1)
    fpr  = 100.0 * fp / max(fp + tn, 1)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    if auc > best_auc:
        best_auc, patience_count = auc, 0
        torch.save(model.state_dict(), f'{WORKING_DIR}/corrosion_model_best.pth')
    else:
        patience_count += 1

    history.append(dict(epoch=epoch, tr_loss=tr_loss, val_loss=val_loss,
                        auc=auc, f1=f1, sens=sens, fpr=fpr))

    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:4d}  {tr_loss:8.5f}  {val_loss:8.5f}  {auc:8.4f}  '
              f'{f1:7.4f}  {sens:6.1f}%  {fpr:5.1f}%  {lr:10.2e}')

    if patience_count >= PATIENCE:
        print(f'Early stop at epoch {epoch}')
        break

print(f'\nBest Val AUC: {best_auc:.4f}')

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

model.load_state_dict(torch.load(f'{WORKING_DIR}/corrosion_model_best.pth', map_location=DEVICE))
model.eval()

logits_v, tgts_v = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits_v.append(model(xb.to(DEVICE)).cpu().numpy())
        tgts_v.append(yb.numpy())

logits_v = np.concatenate(logits_v)
tgts_v   = np.concatenate(tgts_v)
probs_v  = 1.0 / (1.0 + np.exp(-logits_v))
preds_v  = (probs_v >= 0.5).astype(int)

auc = roc_auc_score(tgts_v, probs_v)
f1  = f1_score(tgts_v, preds_v, zero_division=0)
tp  = float(((preds_v == 1) & (tgts_v == 1)).sum())
fn  = float(((preds_v == 0) & (tgts_v == 1)).sum())
fp  = float(((preds_v == 1) & (tgts_v == 0)).sum())
tn  = float(((preds_v == 0) & (tgts_v == 0)).sum())
sens = 100.0 * tp / max(tp + fn, 1)
fpr  = 100.0 * fp / max(fp + tn, 1)

print(f'Val AUC: {auc:.4f}')
print(f'Val F1:  {f1:.4f}')
print(f'Sensitivity: {sens:.1f}%  FPR: {fpr:.1f}%')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC curve
fpr_curve, tpr_curve, _ = roc_curve(tgts_v, probs_v)
axes[0].plot(fpr_curve, tpr_curve, lw=2)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('Sensitivity (TPR)')
axes[0].set_title(f'ROC Curve  AUC={auc:.4f}')
axes[0].axvline(0.3, color='r', lw=1, linestyle='--', label='FPR=30%')
axes[0].legend()

# Score histogram
axes[1].hist(probs_v[tgts_v == 0], bins=50, alpha=0.6, label='healthy (0)', density=True)
axes[1].hist(probs_v[tgts_v == 1], bins=50, alpha=0.6, label='at-risk (1)', density=True)
axes[1].axvline(0.5, color='k', lw=1, linestyle='--')
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Score distributions')
axes[1].legend()

# Loss curves
ep_h = [h['epoch'] for h in history]
axes[2].plot(ep_h, [h['tr_loss']  for h in history], label='Train')
axes[2].plot(ep_h, [h['val_loss'] for h in history], label='Val')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('BCEWithLogitsLoss')
axes[2].set_title('Training curves')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/corrosion_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
import shutil
shutil.copy(f'{WORKING_DIR}/corrosion_model_best.pth', f'{WORKING_DIR}/corrosion_model.pth')
print(f'Saved: {WORKING_DIR}/corrosion_model.pth')
print(f'AUC:         {auc:.4f}')
print(f'F1:          {f1:.4f}')
print(f'Sensitivity: {sens:.1f}%')
print(f'FPR:         {fpr:.1f}%')